<a href="https://colab.research.google.com/github/sinayavarian/github-practice/blob/main/Task7_AutoFeatureSelector.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Task 7: AutoFeatureSelector Tool
## This task is to test your understanding of various Feature Selection methods outlined in the lecture and the ability to apply this knowledge in a real-world dataset to select best features and also to build an automated feature selection tool as your toolkit

### Use your knowledge of different feature selector methods to build an Automatic Feature Selection tool
- Pearson Correlation
- Chi-Square
- RFE
- Embedded
- Tree (Random Forest)
- Tree (Light GBM)

### Dataset: FIFA 19 Player Skills
#### Attributes: FIFA 2019 players attributes like Age, Nationality, Overall, Potential, Club, Value, Wage, Preferred Foot, International Reputation, Weak Foot, Skill Moves, Work Rate, Position, Jersey Number, Joined, Loaned From, Contract Valid Until, Height, Weight, LS, ST, RS, LW, LF, CF, RF, RW, LAM, CAM, RAM, LM, LCM, CM, RCM, RM, LWB, LDM, CDM, RDM, RWB, LB, LCB, CB, RCB, RB, Crossing, Finishing, Heading, Accuracy, ShortPassing, Volleys, Dribbling, Curve, FKAccuracy, LongPassing, BallControl, Acceleration, SprintSpeed, Agility, Reactions, Balance, ShotPower, Jumping, Stamina, Strength, LongShots, Aggression, Interceptions, Positioning, Vision, Penalties, Composure, Marking, StandingTackle, SlidingTackle, GKDiving, GKHandling, GKKicking, GKPositioning, GKReflexes, and Release Clause.

In [1]:
%matplotlib inline
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import scipy.stats as ss
from collections import Counter
import math
from scipy import stats

In [5]:
from google.colab import files
uploaded = files.upload()
player_df = pd.read_csv("fifa19.csv")

Saving fifa19.csv to fifa19.csv


In [6]:
numcols = ['Overall', 'Crossing','Finishing',  'ShortPassing',  'Dribbling','LongPassing', 'BallControl', 'Acceleration','SprintSpeed', 'Agility',  'Stamina','Volleys','FKAccuracy','Reactions','Balance','ShotPower','Strength','LongShots','Aggression','Interceptions']
catcols = ['Preferred Foot','Position','Body Type','Nationality','Weak Foot']

In [7]:
player_df = player_df[numcols+catcols]

In [8]:
traindf = pd.concat([player_df[numcols], pd.get_dummies(player_df[catcols])],axis=1)
features = traindf.columns

traindf = traindf.dropna()

In [9]:
traindf = pd.DataFrame(traindf,columns=features)

In [10]:
y = traindf['Overall']>=87
X = traindf.copy()
del X['Overall']

In [11]:
X.head()

,Crossing,Finishing,ShortPassing,Dribbling,LongPassing,BallControl,Acceleration,SprintSpeed,Agility,Stamina,...,Nationality_Uganda,Nationality_Ukraine,Nationality_United Arab Emirates,Nationality_United States,Nationality_Uruguay,Nationality_Uzbekistan,Nationality_Venezuela,Nationality_Wales,Nationality_Zambia,Nationality_Zimbabwe
0,84.0,95.0,90.0,97.0,87.0,96.0,91.0,86.0,91.0,72.0,...,False,False,False,False,False,False,False,False,False,False
1,84.0,94.0,81.0,88.0,77.0,94.0,89.0,91.0,87.0,88.0,...,False,False,False,False,False,False,False,False,False,False
2,79.0,87.0,84.0,96.0,78.0,95.0,94.0,90.0,96.0,81.0,...,False,False,False,False,False,False,False,False,False,False
3,17.0,13.0,50.0,18.0,51.0,42.0,57.0,58.0,60.0,43.0,...,False,False,False,False,False,False,False,False,False,False
4,93.0,82.0,92.0,86.0,91.0,91.0,78.0,76.0,79.0,90.0,...,False,False,False,False,False,False,False,False,False,False


In [12]:
len(X.columns)

223

### Set some fixed set of features

In [15]:
feature_name = list(X.columns)
# no of maximum features we need to select
num_feats=30

## Filter Feature Selection - Pearson Correlation

### Pearson Correlation function

In [33]:
def cor_selector(X, y, num_feats):
    # convert target to numeric (True/False -> 1/0)
    y = y.astype(int)

    # compute Pearson correlation between each feature and target
    cor_list = []
    for i in X.columns:
        cor = X[i].corr(y)
        cor_list.append(cor)

    # replace NaN with 0
    cor_list = [0 if np.isnan(i) else i for i in cor_list]

    # take absolute value and sort
    cor_abs = np.abs(cor_list)

    # select top num_feats features
    cor_feature = X.columns[np.argsort(cor_abs)[-num_feats:]].tolist()

    # support mask (True for selected features)
    cor_support = [True if i in cor_feature else False for i in X.columns]

    return cor_support, cor_feature


In [34]:
cor_support, cor_feature = cor_selector(X, y,num_feats)
print(str(len(cor_feature)), 'selected features')

30 selected features


### List the selected features from Pearson Correlation

In [35]:
cor_feature

['Nationality_Costa Rica',
 'Position_LAM',
 'Nationality_Uruguay',
 'Acceleration',
 'SprintSpeed',
 'Strength',
 'Nationality_Gabon',
 'Nationality_Slovenia',
 'Stamina',
 'Weak Foot',
 'Agility',
 'Crossing',
 'Nationality_Belgium',
 'Dribbling',
 'ShotPower',
 'LongShots',
 'Finishing',
 'BallControl',
 'FKAccuracy',
 'LongPassing',
 'Volleys',
 'ShortPassing',
 'Position_RF',
 'Position_LF',
 'Body Type_PLAYER_BODY_TYPE_25',
 'Body Type_Courtois',
 'Body Type_Neymar',
 'Body Type_Messi',
 'Body Type_C. Ronaldo',
 'Reactions']

## Filter Feature Selection - Chi-Sqaure

In [36]:
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import chi2
from sklearn.preprocessing import MinMaxScaler

### Chi-Squared Selector function

In [37]:
def chi_squared_selector(X, y, num_feats):
    # Chi-square requires non-negative features
    X_ = X.copy()

    # Convert target to numeric
    y_ = y.astype(int)

    # Apply Chi-Square feature selection
    selector = SelectKBest(score_func=chi2, k=num_feats)
    selector.fit(X_, y_)

    # Get support mask and selected feature names
    chi_support = selector.get_support()
    chi_feature = X.columns[chi_support].tolist()

    return chi_support, chi_feature

In [38]:
chi_support, chi_feature = chi_squared_selector(X, y,num_feats)
print(str(len(chi_feature)), 'selected features')

30 selected features


### List the selected features from Chi-Square

In [39]:
chi_feature

['Crossing',
 'Finishing',
 'ShortPassing',
 'Dribbling',
 'LongPassing',
 'BallControl',
 'Acceleration',
 'SprintSpeed',
 'Agility',
 'Stamina',
 'Volleys',
 'FKAccuracy',
 'Reactions',
 'Balance',
 'ShotPower',
 'Strength',
 'LongShots',
 'Aggression',
 'Interceptions',
 'Position_LF',
 'Position_RF',
 'Body Type_C. Ronaldo',
 'Body Type_Courtois',
 'Body Type_Messi',
 'Body Type_Neymar',
 'Body Type_PLAYER_BODY_TYPE_25',
 'Nationality_Belgium',
 'Nationality_Gabon',
 'Nationality_Slovenia',
 'Nationality_Uruguay']

:## Wrapper Feature Selection - Recursive Feature Elimination

In [40]:
from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import MinMaxScaler

### RFE Selector function

In [41]:
def rfe_selector(X, y, num_feats):
    # y: True/False -> 1/0
    y_ = y.astype(int)

    # estimator must provide coef_ (LogisticRegression does)
    estimator = LogisticRegression(max_iter=2000, solver="liblinear")

    # RFE: recursively remove the least important features until k remain
    selector = RFE(estimator=estimator, n_features_to_select=num_feats, step=1)
    selector.fit(X, y_)

    rfe_support = selector.get_support()                 # boolean mask
    rfe_feature = X.columns[rfe_support].tolist()        # selected names

    return rfe_support, rfe_feature

In [42]:
rfe_support, rfe_feature = rfe_selector(X, y,num_feats)
print(str(len(rfe_feature)), 'selected features')

30 selected features


### List the selected features from RFE

In [43]:
rfe_feature

['Weak Foot',
 'Preferred Foot_Left',
 'Preferred Foot_Right',
 'Position_CM',
 'Position_LAM',
 'Position_LF',
 'Position_LM',
 'Position_LW',
 'Position_RB',
 'Position_RF',
 'Position_RW',
 'Body Type_C. Ronaldo',
 'Body Type_Lean',
 'Body Type_Normal',
 'Body Type_PLAYER_BODY_TYPE_25',
 'Body Type_Stocky',
 'Nationality_Belgium',
 'Nationality_Chile',
 'Nationality_Costa Rica',
 'Nationality_Croatia',
 'Nationality_England',
 'Nationality_France',
 'Nationality_Gabon',
 'Nationality_Japan',
 'Nationality_Korea Republic',
 'Nationality_Netherlands',
 'Nationality_Slovakia',
 'Nationality_Slovenia',
 'Nationality_Sweden',
 'Nationality_Uruguay']

## Embedded Selection - Lasso: SelectFromModel

In [44]:
from sklearn.feature_selection import SelectFromModel
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import MinMaxScaler

In [46]:
def embedded_log_reg_selector(X, y, num_feats):
    # Convert target to numeric
    y_ = y.astype(int)

    # L1-regularized Logistic Regression
    log_reg = LogisticRegression(
        penalty='l1',
        solver='liblinear',
        C=0.1,          # regularization strength
        max_iter=2000
    )

    # Embedded feature selection
    selector = SelectFromModel(
        log_reg,
        max_features=num_feats
    )

    selector.fit(X, y_)

    embedded_lr_support = selector.get_support()
    embedded_lr_feature = X.columns[embedded_lr_support].tolist()

    return embedded_lr_support, embedded_lr_feature


In [47]:
embedded_lr_support, embedded_lr_feature = embedded_log_reg_selector(X, y, num_feats)
print(str(len(embedded_lr_feature)), 'selected features')

19 selected features


In [48]:
embedded_lr_feature

['Crossing',
 'Finishing',
 'Dribbling',
 'LongPassing',
 'BallControl',
 'Acceleration',
 'SprintSpeed',
 'Agility',
 'Stamina',
 'Volleys',
 'FKAccuracy',
 'Reactions',
 'Balance',
 'ShotPower',
 'Strength',
 'Aggression',
 'Interceptions',
 'Weak Foot',
 'Body Type_Lean']

## Tree based(Random Forest): SelectFromModel

In [49]:
from sklearn.feature_selection import SelectFromModel
from sklearn.ensemble import RandomForestClassifier

In [50]:
def embedded_rf_selector(X, y, num_feats):
    # y: True/False -> 1/0
    y_ = y.astype(int)

    # Tree-based model (embedded feature importance)
    rf = RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        n_jobs=-1
    )

    # Select features using model importances
    selector = SelectFromModel(
        estimator=rf,
        max_features=num_feats,   # upper bound
        threshold=-float("inf")   # allows selecting up to max_features
    )

    selector.fit(X, y_)

    embedded_rf_support = selector.get_support()                 # boolean mask
    embedded_rf_feature = X.columns[embedded_rf_support].tolist()# selected names

    return embedded_rf_support, embedded_rf_feature

In [62]:
embedded_rf_support, embedded_rf_feature = embedded_rf_selector(X, y, num_feats)
print(str(len(embedded_rf_feature)), 'selected features')

30 selected features


In [54]:
embedded_rf_feature

['Crossing',
 'Finishing',
 'ShortPassing',
 'Dribbling',
 'LongPassing',
 'BallControl',
 'Acceleration',
 'SprintSpeed',
 'Agility',
 'Stamina',
 'Volleys',
 'FKAccuracy',
 'Reactions',
 'Balance',
 'ShotPower',
 'Strength',
 'LongShots',
 'Aggression',
 'Interceptions',
 'Weak Foot',
 'Preferred Foot_Left',
 'Preferred Foot_Right',
 'Position_ST',
 'Body Type_Courtois',
 'Body Type_Lean',
 'Body Type_Normal',
 'Nationality_Belgium',
 'Nationality_Brazil',
 'Nationality_Italy',
 'Nationality_Slovenia']

## Tree based(Light GBM): SelectFromModel

In [55]:
from sklearn.feature_selection import SelectFromModel
from lightgbm import LGBMClassifier

In [73]:
def embedded_lgbm_selector(X, y, num_feats):
    X_ = X.copy()
    X_.columns = X_.columns.str.replace(r"\s+", "_", regex=True)

    y_ = y.astype(int)

    pos = (y_ == 1).sum()
    neg = (y_ == 0).sum()
    spw = neg / pos

    lgbm = LGBMClassifier(
        n_estimators=500,
        learning_rate=0.05,
        scale_pos_weight=spw,
        max_depth=6,
        num_leaves=31,
        random_state=42,
        n_jobs=-1
    )



    selector = SelectFromModel(
        estimator=lgbm,
        max_features=num_feats,
        threshold=-float("inf")
    )

    selector.fit(X_, y_)
    embedded_lgbm_support = selector.get_support()
    embedded_lgbm_feature = X_.columns[embedded_lgbm_support].tolist()

    return embedded_lgbm_support, embedded_lgbm_feature

In [74]:
embedded_lgbm_support, embedded_lgbm_feature = embedded_lgbm_selector(X, y, num_feats)
print(str(len(embedded_lgbm_feature)), 'selected features')

[LightGBM] [Info] Number of positive: 55, number of negative: 18104
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.037418 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1812
[LightGBM] [Info] Number of data points in the train set: 18159, number of used features: 124
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.003029 -> initscore=-5.796555
[LightGBM] [Info] Start training from score -5.796555
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain:

In [58]:
embedded_lgbm_feature

['Crossing',
 'Finishing',
 'ShortPassing',
 'Dribbling',
 'LongPassing',
 'BallControl',
 'Acceleration',
 'SprintSpeed',
 'Agility',
 'Stamina',
 'Volleys',
 'FKAccuracy',
 'Reactions',
 'Balance',
 'ShotPower',
 'Strength',
 'LongShots',
 'Aggression',
 'Interceptions',
 'Weak Foot',
 'Preferred Foot_Left',
 'Position_CB',
 'Position_LCB',
 'Body Type_Lean',
 'Body Type_Normal',
 'Nationality_France',
 'Nationality_Germany',
 'Nationality_Italy',
 'Nationality_Slovenia',
 'Nationality_Spain']

## Putting all of it together: AutoFeatureSelector Tool

In [77]:
pd.set_option('display.max_rows', None)

feature_selection_df = pd.DataFrame({
    'Feature': feature_name,
    'Pearson': cor_support,
    'Chi-2': chi_support,
    'RFE': rfe_support,
    'Logistics': embedded_lr_support,
    'Random Forest': embedded_rf_support,
    'LightGBM': embedded_lgbm_support
})

# columns to vote
selection_cols = ['Pearson','Chi-2','RFE','Logistics','Random Forest','LightGBM']

feature_selection_df[selection_cols] = feature_selection_df[selection_cols].astype(int)

# count votes (True=1, False=0)
feature_selection_df['Total'] = feature_selection_df[selection_cols].sum(axis=1)

# sort & show top num_feats
feature_selection_df = feature_selection_df.sort_values(['Total','Feature'], ascending=False)
feature_selection_df.index = range(1, len(feature_selection_df)+1)

feature_selection_df.head(num_feats)


,Feature,Pearson,Chi-2,RFE,Logistics,Random Forest,LightGBM,Total
1,Weak Foot,1,0,1,1,1,1,5
2,Volleys,1,1,0,1,1,1,5
3,Strength,1,1,0,1,1,1,5
4,Stamina,1,1,0,1,1,1,5
5,SprintSpeed,1,1,0,1,1,1,5
6,ShotPower,1,1,0,1,1,1,5
7,Reactions,1,1,0,1,1,1,5
8,Nationality_Belgium,1,1,1,0,1,1,5
9,LongPassing,1,1,0,1,1,1,5
10,Finishing,1,1,0,1,1,1,5


## Can you build a Python script that takes dataset and a list of different feature selection methods that you want to try and output the best (maximum votes) features from all methods?

In [78]:

import numpy as np
import pandas as pd

from sklearn.feature_selection import SelectKBest, chi2, RFE, SelectFromModel
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler

# Optional LightGBM
try:
    from lightgbm import LGBMClassifier
    HAS_LGBM = True
except Exception:
    HAS_LGBM = False


# ---------------------------
# Feature selection methods
# ---------------------------
def cor_selector(X, y, num_feats):
    y_ = y.astype(int)
    cor_list = []
    for col in X.columns:
        cor = X[col].corr(y_)
        cor_list.append(0 if np.isnan(cor) else cor)

    cor_abs = np.abs(cor_list)
    cor_feature = X.columns[np.argsort(cor_abs)[-num_feats:]].tolist()
    cor_support = np.array([c in cor_feature for c in X.columns], dtype=bool)
    return cor_support, cor_feature


def chi_squared_selector(X, y, num_feats):
    # chi2 requires non-negative features
    X_ = X.copy()
    min_vals = X_.min()
    neg_cols = min_vals[min_vals < 0].index
    if len(neg_cols) > 0:
        X_[neg_cols] = X_[neg_cols].sub(min_vals[neg_cols], axis=1)

    y_ = y.astype(int)
    selector = SelectKBest(score_func=chi2, k=num_feats)
    selector.fit(X_, y_)
    chi_support = selector.get_support()
    chi_feature = X.columns[chi_support].tolist()
    return chi_support, chi_feature


def rfe_selector(X, y, num_feats):
    y_ = y.astype(int)
    estimator = LogisticRegression(max_iter=2000, solver="liblinear")
    selector = RFE(estimator=estimator, n_features_to_select=num_feats, step=1)
    selector.fit(X, y_)
    rfe_support = selector.get_support()
    rfe_feature = X.columns[rfe_support].tolist()
    return rfe_support, rfe_feature


def embedded_log_reg_selector(X, y, num_feats, C=0.1, threshold=0.0):
    # L1 Logistic Regression (Lasso-style) -> embedded selection
    y_ = y.astype(int)
    log_reg = LogisticRegression(
        penalty="l1",
        solver="liblinear",
        C=C,
        max_iter=2000
    )
    selector = SelectFromModel(
        estimator=log_reg,
        max_features=num_feats,
        threshold=threshold
    )
    selector.fit(X, y_)
    support = selector.get_support()
    feats = X.columns[support].tolist()
    return support, feats


def embedded_rf_selector(X, y, num_feats, n_estimators=300):
    y_ = y.astype(int)
    rf = RandomForestClassifier(
        n_estimators=n_estimators,
        random_state=42,
        n_jobs=-1
    )
    selector = SelectFromModel(
        estimator=rf,
        max_features=num_feats,
        threshold=-float("inf")
    )
    selector.fit(X, y_)
    support = selector.get_support()
    feats = X.columns[support].tolist()
    return support, feats


def embedded_lgbm_selector(X, y, num_feats, n_estimators=500, learning_rate=0.05):
    if not HAS_LGBM:
        raise ImportError("LightGBM is not installed. Run: pip install lightgbm")

    X_ = X.copy()
    # Replace whitespace in feature names (LightGBM warning fix)
    X_.columns = X_.columns.str.replace(r"\s+", "_", regex=True)

    y_ = y.astype(int)
    pos = (y_ == 1).sum()
    neg = (y_ == 0).sum()
    spw = (neg / pos) if pos > 0 else 1.0

    lgbm = LGBMClassifier(
        n_estimators=n_estimators,
        learning_rate=learning_rate,
        scale_pos_weight=spw,
        random_state=42,
        n_jobs=-1
    )

    selector = SelectFromModel(
        estimator=lgbm,
        max_features=num_feats,
        threshold=-float("inf")
    )
    selector.fit(X_, y_)

    support = selector.get_support()
    feats = X_.columns[support].tolist()
    return support, feats


# ---------------------------
# Preprocessing (FIFA-style)
# ---------------------------
def preprocess_dataset(dataset_path):

    player_df = pd.read_csv(dataset_path)

    numcols = [
        'Overall', 'Crossing', 'Finishing', 'ShortPassing', 'Dribbling',
        'LongPassing', 'BallControl', 'Acceleration', 'SprintSpeed', 'Agility',
        'Stamina', 'Volleys', 'FKAccuracy', 'Reactions', 'Balance', 'ShotPower',
        'Strength', 'LongShots', 'Aggression', 'Interceptions'
    ]
    catcols = ['Preferred Foot', 'Position', 'Body Type', 'Nationality', 'Weak Foot']

    # Keep only needed columns
    player_df = player_df[numcols + catcols]

    # One-hot encode categoricals and join to numeric features
    traindf = pd.concat([player_df[numcols], pd.get_dummies(player_df[catcols])], axis=1)
    traindf = traindf.dropna()

    # Define target and features
    y = traindf['Overall'] >= 87
    X = traindf.copy()
    del X['Overall']

    # How many features we want to select (can be changed)
    num_feats = 30
    return X, y, num_feats



In [79]:

def autoFeatureSelector(dataset_path, methods=None, num_feats_override=None, min_votes=1):

    if methods is None:
        methods = ['pearson', 'chi-square', 'rfe', 'log-reg', 'rf', 'lgbm']

    X, y, num_feats = preprocess_dataset(dataset_path)
    if num_feats_override is not None:
        num_feats = int(num_feats_override)

    # Prepare scaled data for linear methods (RFE, L1-logreg)
    scaler = StandardScaler()
    X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns, index=X.index)

    # Run selected methods
    results = {}  # method_name -> support mask (bool array aligned to X.columns)

    if 'pearson' in methods:
        cor_support, _ = cor_selector(X, y, num_feats)
        results['Pearson'] = cor_support

    if 'chi-square' in methods:
        chi_support, _ = chi_squared_selector(X, y, num_feats)
        results['Chi-2'] = chi_support

    if 'rfe' in methods:
        rfe_support, _ = rfe_selector(X_scaled, y, num_feats)
        results['RFE'] = rfe_support

    if 'log-reg' in methods:
        embedded_lr_support, _ = embedded_log_reg_selector(X_scaled, y, num_feats, C=0.1, threshold=0.0)
        results['LogReg(L1)'] = embedded_lr_support

    if 'rf' in methods:
        embedded_rf_support, _ = embedded_rf_selector(X, y, num_feats)
        results['Random Forest'] = embedded_rf_support

    if 'lgbm' in methods:
        if not HAS_LGBM:
            raise ImportError("You requested 'lgbm' but lightgbm is not installed. Run: pip install lightgbm")
        embedded_lgbm_support, _ = embedded_lgbm_selector(X, y, num_feats)
        results['LightGBM'] = embedded_lgbm_support

    # Combine votes into a single table
    feature_selection_df = pd.DataFrame({'Feature': list(X.columns)})

    for method_name, support in results.items():
        feature_selection_df[method_name] = support

    vote_cols = list(results.keys())
    feature_selection_df['Total'] = feature_selection_df[vote_cols].sum(axis=1)

    # Sort by votes then feature name for stability
    feature_selection_df = feature_selection_df.sort_values(['Total', 'Feature'], ascending=False).reset_index(drop=True)

    # Select "best" features: maximum votes, then take top num_feats (or fewer if min_votes is high)
    filtered = feature_selection_df[feature_selection_df['Total'] >= min_votes]

    best_features = filtered.head(num_feats)['Feature'].tolist()

    return best_features, feature_selection_df



In [80]:
best_features = autoFeatureSelector(dataset_path="fifa19.csv", methods=['pearson', 'chi-square', 'rfe', 'log-reg', 'rf', 'lgbm'])
best_features

[LightGBM] [Info] Number of positive: 55, number of negative: 18104
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.010120 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1812
[LightGBM] [Info] Number of data points in the train set: 18159, number of used features: 124
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.003029 -> initscore=-5.796555
[LightGBM] [Info] Start training from score -5.796555
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain:

(['ShortPassing',
  'Reactions',
  'LongPassing',
  'Finishing',
  'Crossing',
  'BallControl',
  'Strength',
  'Stamina',
  'SprintSpeed',
  'Nationality_Slovenia',
  'FKAccuracy',
  'Dribbling',
  'Agility',
  'Acceleration',
  'Volleys',
  'ShotPower',
  'Nationality_Belgium',
  'LongShots',
  'Body Type_Lean',
  'Body Type_Courtois',
  'Aggression',
  'Weak Foot',
  'Position_RF',
  'Position_LF',
  'Nationality_Uruguay',
  'Nationality_Gabon',
  'Interceptions',
  'Body Type_PLAYER_BODY_TYPE_25',
  'Body Type_Normal',
  'Body Type_Neymar'],
                               Feature  Pearson  Chi-2    RFE  LogReg(L1)  \
 0                        ShortPassing     True   True   True        True   
 1                           Reactions     True   True   True        True   
 2                         LongPassing     True   True   True        True   
 3                           Finishing     True   True   True        True   
 4                            Crossing     True   True   True  

### Last, Can you turn this notebook into a python script, run it and submit the python (.py) file that takes dataset and list of methods as inputs and outputs the best features